<a href="https://colab.research.google.com/github/henriquecrispim/live-finance-dashboard/blob/main/live_finance_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# DASHBOARD MULTI-MERCADO INTERATIVO (HORÁRIO DE BRASÍLIA)
# ==============================================================================

# 1. Instalação e Configuração das Dependências
!pip install pandas yfinance plotly --quiet

import yfinance as yf
import pandas as pd
import plotly.express as px
import plotly.io as pio
import time
from datetime import datetime, timezone, timedelta  # Inclusão para controle de fuso horário
from IPython.display import clear_output, display

# Usando o renderizador nativo padrão do Colab para gráficos interativos
pio.renderers.default = 'colab'

tickers = [
    "PETR4.SA", "VALE3.SA", "ITUB4.SA", "MGLU3.SA",  # Ações Nacionais (R$)
    "HGLG11.SA",                                    # Fundo Imobiliário (R$)
    "AAPL",                                         # Ação Americana (US$)
    "BTC-USD", "ETH-USD",                           # Criptomoedas (US$)
    "BRL=X"                                         # Câmbio USD/BRL
]

print("[INFO] Conectando à esteira de dados inteligente...")
time.sleep(2)

# Configuração explícita do Fuso Horário de Brasília (GMT -3)
fuso_brasilia = timezone(timedelta(hours=-3))

try:
    while True:
        # 2. CAPTURA INTELIGENTE DE DADOS
        dados_mercado = yf.download(tickers, period="1d", interval="1m", auto_adjust=True, progress=False)["Close"]

        # Pega a última linha válida (preenche dados em branco se o mercado fechou)
        dados_limpos = dados_mercado.ffill().bfill()
        precos_atuais = dados_limpos.iloc[-1].to_dict()

        # Captura o Dólar (se falhar por mercado fechado, assume um fallback estável de 5.50)
        usd_brl = precos_atuais.get("BRL=X", 5.50)
        if pd.isna(usd_brl): usd_brl = 5.50

        # 3. FUNÇÃO DE CHECAGEM (Garante que nenhum valor seja 'nan')
        def obter_preco_safe(ticker_name, padrao=1.0):
            val = precos_atuais.get(ticker_name, padrao)
            return padrao if pd.isna(val) else val

        carteira = [
            {'Ativo': 'Petrobras (PETR4)', 'Valor_Total': obter_preco_safe('PETR4.SA', 37.50) * 100},
            {'Ativo': 'Vale (VALE3)', 'Valor_Total': obter_preco_safe('VALE3.SA', 78.00) * 50},
            {'Ativo': 'Itaú (ITUB4)', 'Valor_Total': obter_preco_safe('ITUB4.SA', 42.00) * 120},
            {'Ativo': 'Magalu (MGLU3)', 'Valor_Total': obter_preco_safe('MGLU3.SA', 4.30) * 300},
            {'Ativo': 'FII HGLG11', 'Valor_Total': obter_preco_safe('HGLG11.SA', 165.00) * 40},
            {'Ativo': 'Apple (AAPL) [US$]', 'Valor_Total': (obter_preco_safe('AAPL', 210.00) * usd_brl) * 15},
            {'Ativo': 'Bitcoin (BTC) [Cripto]', 'Valor_Total': (obter_preco_safe('BTC-USD', 61000.0) * usd_brl) * 0.05},
            {'Ativo': 'Ethereum (ETH) [Cripto]', 'Valor_Total': (obter_preco_safe('ETH-USD', 3400.0) * usd_brl) * 0.4}
        ]

        df_atualizado = pd.DataFrame(carteira)

        # LIMPA A TELA APENAS QUANDO O PRÓXIMO PASSO ESTIVER PRONTO
        clear_output(wait=True)

        # Captura o momento exato e força a conversão para o fuso horário de Brasília
        agora_brasilia = datetime.now(fuso_brasilia)
        horario_atual = agora_brasilia.strftime('%H:%M:%S')

        # Painel Informativo Textual no Topo
        print(f"🔄 [SISTEMA RESILIENTE ACTIVE] Atualizado às: {horario_atual} (Horário de Brasília) | Câmbio: 1 US$ = R$ {usd_brl:.2f}")
        print("-" * 75)
        print("Nota: Se o mercado tradicional estiver fechado, o sistema usa o último preço válido.")
        print("-" * 75)

        # ==============================================================================
        # 4. ENGENHARIA GRÁFICA INTERATIVA NATIVAMENTE COMPATÍVEL
        # ==============================================================================
        fig = px.pie(
            df_atualizado,
            values='Valor_Total',
            names='Ativo',
            hole=0.60,
            color_discrete_sequence=px.colors.qualitative.Prism,
            title=f"Dashboard Multi-Mercado Live — Patrimônio Unificado em R$ às {horario_atual} (BRT)"
        )

        fig.update_traces(
            textinfo='percent+value',
            texttemplate='<b>%{percent}</b><br>R$ %{value:,.2f}',
            hovertemplate="<b>Ativo:</b> %{label}<br><b>Posição:</b> R$ %{value:,.2f}<br><extra></extra>"
        )

        fig.update_layout(
            title=dict(font=dict(size=16, family="Arial", color="#1a252f"), pad=dict(t=15, b=15)),
            margin={"r":20,"t":60,"l":20,"b":20},
            legend=dict(title_text="<b>Alocação de Ativos</b>", yanchor="middle", y=0.5, xanchor="left", x=1.05)
        )

        # Salva o arquivo HTML na barra lateral de forma invisível
        fig.write_html("dashboard_financas_reais.html")

        # Exibe o gráfico interativo
        fig.show()

        # Espera 10 segundos para o próximo pulso
        time.sleep(10)

except KeyboardInterrupt:
    print("\n[INFO] Monitoramento multi-mercado encerrado com segurança.")

🔄 [SISTEMA RESILIENTE ACTIVE] Atualizado às: 10:24:22 (Horário de Brasília) | Câmbio: 1 US$ = R$ 5.15
---------------------------------------------------------------------------
Nota: Se o mercado tradicional estiver fechado, o sistema usa o último preço válido.
---------------------------------------------------------------------------



[INFO] Monitoramento multi-mercado encerrado com segurança.
